In [6]:
import os


In [5]:
# 1. 定义文件路径
meme_file_path = "/lulabdata3/huangkeyun/zhangys/RNA_locator/motif_analysis/motif_databases/CISBP-RNA/Homo_sapiens.meme"

# 2. 定义目标 RBP 列表 (合并了原始列表与小鼠源补充列表，并进行了关键词去重)
ev_rbps = [
    '(RBM38)_(Mus_musculus)_(RBD_0.99)', 'A2BP1', 'CSDA', 'ELAVL2', 'HNRNPC', 'SNRPA', 'TARDBP', 'hnRNPK',
    '(Elavl2)_(Homo_sapiens)_(RBD_1.00)', '(Hnrnpc)_(Homo_sapiens)_(RBD_1.00)', '(Rbfox1)_(Homo_sapiens)_(RBD_1.00)', 
    '(Snrpa)_(Homo_sapiens)_(RBD_0.98)', '(Tardbp)_(Homo_sapiens)_(RBD_0.92)', '(Ybx1)_(Homo_sapiens)_(RBD_1.00)', 
    '(Zc3h10)_(Homo_sapiens)_(RBD_1.00)', 'Rbm38', 'YBX1'
]

cyto_rbps = [
    '(RBM45)_(Mus_musculus)_(RBD_0.94)', 'CPEB2', 'FMR1', 'HNRNPL', 'LIN28A', 'NOVA2', 'PABPN1', 'PPRC1', 
    'RBM3', 'RBM8A', 'SAMD4A', 'SRSF7', 'STAR-PAP', 'hnRNPLL',
    '(Cpeb2)_(Homo_sapiens)_(RBD_1.00)', '(Fmr1)_(Homo_sapiens)_(RBD_0.97)', '(Hnrpll)_(Homo_sapiens)_(RBD_0.99)', 
    '(Lin28a)_(Homo_sapiens)_(RBD_0.99)', '(Lin28b)_(Homo_sapiens)_(RBD_0.83)', '(Nova1)_(Homo_sapiens)_(RBD_0.91)', 
    '(Pabpn1)_(Homo_sapiens)_(RBD_1.00)', '(Pprc1)_(Homo_sapiens)_(RBD_1.00)', '(Rbm3)_(Homo_sapiens)_(RBD_0.99)', 
    '(Rbm8a)_(Homo_sapiens)_(RBD_1.00)', '(Samd4)_(Homo_sapiens)_(RBD_0.95)', '(Tut1)_(Homo_sapiens)_(RBD_0.85)', 
    'Rbm45', 'LIN28B', 'NOVA1'
]

shared_rbps = [
    '(HNRNPR)_(Gallus_gallus)_(RBD_0.97)', '(PCBP3)_(Mus_musculus)_(RBD_1.00)', 'CPEB4', 'HNRNPCL1', 'MBNL1', 
    'PCBP1', 'PCBP2', 'PTBP1', 'RALY', 'ROD1', 'TIA1', 'U2AF2', 'YB-1', 'ZC3H10',
    '(Cpeb4)_(Homo_sapiens)_(RBD_1.00)', '(Csda)_(Homo_sapiens)_(RBD_1.00)', '(Hnrnpk)_(Homo_sapiens)_(RBD_1.00)', 
    '(Mbnl1)_(Homo_sapiens)_(RBD_1.00)', '(Pcbp2)_(Homo_sapiens)_(RBD_1.00)', '(Ptbp1)_(Homo_sapiens)_(RBD_0.96)', 
    '(Raly)_(Homo_sapiens)_(RBD_0.97)', '(Rod1)_(Homo_sapiens)_(RBD_0.80)', '(Tia1)_(Homo_sapiens)_(RBD_1.00)', 
    '(Tial1)_(Homo_sapiens)_(RBD_0.89)', '(U2af2)_(Homo_sapiens)_(RBD_1.00)', 'Pcbp1', 'Pcbp3', 'TIAL1'
]

# 辅助函数：根据概率矩阵计算 Consensus Sequence
def get_consensus(matrix_lines, alphabet="ACGU"):
    consensus = ""
    for line in matrix_lines:
        try:
            probs = list(map(float, line.strip().split()))
            if not probs: continue
            max_idx = probs.index(max(probs))
            consensus += alphabet[max_idx]
        except ValueError:
            continue
    return consensus

# 3. 解析 MEME 文件
def extract_meme_data(input_file, target_list, output_file, group_name):
    # 将 target_list 转换为大写集合以进行不加区分的匹配
    targets_upper = [t.upper() for t in target_list]
    
    with open(input_file, 'r') as f:
        lines = f.readlines()

    header_lines = []
    motif_blocks = []
    found_count = 0
    
    in_header = True
    current_motif_lines = []
    keep_current_motif = False
    current_motif_name = ""
    matrix_lines = []
    in_matrix = False

    for line in lines:
        if line.startswith("MOTIF"):
            in_header = False
            # 如果上一个块被标记为保留，处理它
            if keep_current_motif:
                motif_blocks.append("".join(current_motif_lines))
                consensus = get_consensus(matrix_lines)
                print(f"[{group_name}] 找到: {current_motif_name.ljust(45)} | Consensus: {consensus}")
                found_count += 1
            
            # 初始化新的块
            current_motif_lines = [line]
            matrix_lines = []
            in_matrix = False
            
            # 匹配检查（不区分大小写）
            line_upper = line.upper()
            keep_current_motif = any(t in line_upper for t in targets_upper)
            
            if keep_current_motif:
                parts = line.strip().split()
                current_motif_name = parts[-1] if len(parts) > 1 else parts[0]
                
        elif in_header:
            header_lines.append(line)
        else:
            current_motif_lines.append(line)
            if line.startswith("letter-probability matrix:"):
                in_matrix = True
            elif line.startswith("URL") or line.strip() == "":
                in_matrix = False
            elif in_matrix:
                matrix_lines.append(line)

    # 检查最后一个 motif
    if keep_current_motif:
        motif_blocks.append("".join(current_motif_lines))
        consensus = get_consensus(matrix_lines)
        print(f"[{group_name}] 找到: {current_motif_name.ljust(45)} | Consensus: {consensus}")
        found_count += 1

    # 写入文件
    with open(output_file, 'w') as out_f:
        out_f.writelines(header_lines)
        out_f.writelines(motif_blocks)
    
    print(f"统计: {group_name} 匹配到 {found_count} 个 Motif 块。")
    print(f"--- 结果已保存至 {output_file} ---\n")

# 4. 执行提取
print("正在从数据库中提取合并后的 Motif 信息 (包含人源与小鼠源补充)...\n")
extract_meme_data(meme_file_path, ev_rbps, "./meme_files/EV_specific_motifs.meme", "EV独有")
extract_meme_data(meme_file_path, cyto_rbps, "./meme_files/Cyto_specific_motifs.meme", "Cyto独有")
extract_meme_data(meme_file_path, shared_rbps, "./meme_files/Shared_motifs.meme", "共享RBP")

正在从数据库中提取合并后的 Motif 信息 (包含人源与小鼠源补充)...

[EV独有] 找到: HNRNPC                                        | Consensus: AUUUUUU
[EV独有] 找到: hnRNPK                                        | Consensus: CCAACCC
[EV独有] 找到: (RBM38)_(Mus_musculus)_(RBD_0.99)             | Consensus: GUGUGUG
[EV独有] 找到: SNRPA                                         | Consensus: UUGCACA
[EV独有] 找到: TARDBP                                        | Consensus: UGAAUGAG
[EV独有] 找到: CSDA                                          | Consensus: AACAUCA
[EV独有] 找到: HNRNPCL1                                      | Consensus: AUUUUUU
[EV独有] 找到: A2BP1                                         | Consensus: UGCAUGC
[EV独有] 找到: ELAVL2                                        | Consensus: AUCCUUUUUUUUCG
统计: EV独有 匹配到 9 个 Motif 块。
--- 结果已保存至 ./meme_files/EV_specific_motifs.meme ---

[Cyto独有] 找到: CPEB2                                         | Consensus: CUUUUUU
[Cyto独有] 找到: FMR1                                          | Consensus: GGACAAG
[Cyto独有] 找到